# 🚀 Mini Project 1 — Multi-Tool AI Assistant
## Google Colab | LangChain | Search | Weather | PDF RAG | Chat History | LangSmith

**Goal:** Build ONE AI agent that receives a question and automatically chooses:

  **Web Search** → latest/current/recent information

  **Weather** → live weather for a city

  **PDF RAG** → questions about an uploaded PDF

  **Chat History** → only the latest 10 message objects

  **LangSmith** → trace the agent and tool calls


### Architecture

```text
                         USER QUESTION
                                │
                                ▼
                     ┌────────────────────┐
                     │    LLM / AGENT     │
                     │   Tool Selection   │
                     └─────────┬──────────┘
                               │
              ┌────────────────┼────────────────┐
              ▼                ▼                ▼
        🔎 WEB SEARCH      🌤️ WEATHER       📄 PDF RAG
          SerpAPI          OpenWeather       FAISS
              │                │                │
              └────────────────┼────────────────┘
                               ▼
                         TOOL RESULT
                               │
                               ▼
                         FINAL ANSWER

                    + Latest 10 messages
                    + LangSmith tracing
```

# 1. 🧑‍🏫 Development Approach


### Step 1 — LLM Calling
First test a normal LLM call.

### Step 2 — Tools
Build and test Search, Weather, and PDF RAG independently.

### Step 3 — Agent
Give the three tools to one LangChain agent.

### Step 4 — Tool Calling
Understand how the LLM selects a tool and Python executes it.

### Step 5 — Memory
Keep only the latest 10 message objects.

### Step 6 — Tracing
Enable LangSmith and inspect the agent/tool execution.

### Step 7 — End-to-End Tests
Run all assignment-required questions.


# 2. Install Required Packages

We use:

- `langchain` — agent framework
- `langchain-openai` — LLM connection
- `langchain-community` — FAISS and embeddings
- `faiss-cpu` — vector database
- `sentence-transformers` — embeddings
- `pypdf` — PDF extraction
- `requests` — API calls
- `langsmith` — tracing

In [ ]:
# Install packages
# Install project dependencies
!pip install -q -U langchain langchain-openai langchain-community langchain-text-splitters langsmith faiss-cpu sentence-transformers pypdf

print("✅ Packages installed")

# 3. 🔐 Secure API Keys

Do **not** hard-code API keys in the notebook.



### Required keys

| Secret | Purpose |
|---|---|
| `OPENROUTER_API_KEY` | LLM |
| `SERPAPI_API_KEY` | Web Search |
| `OPENWEATHER_API_KEY` | Live Weather |
| `LANGSMITH_API_KEY` | LangSmith |


In [ ]:
import os
from getpass import getpass

def get_secret(name):
    value = os.environ.get(name)

    if value:
        return value

    try:
        from google.colab import userdata
        colab_secret_value = userdata.get(name)
        if colab_secret_value is not None:
            # Explicitly convert to string for os.environ
            value = str(colab_secret_value)
            os.environ[name] = value
            return value
    except Exception:
        pass

    value = getpass(f"Enter {name}: ")
    os.environ[name] = value
    return value


for key in [
    "OPENROUTER_API_KEY",
    "SERPAPI_API_KEY",
    "OPENWEATHER_API_KEY",
    "LANGSMITH_API_KEY",
]:
    get_secret(key)

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "Multi-Tool-AI-Assistant"

print("✅ API keys loaded securely")
print("✅ LangSmith tracing:", os.environ["LANGSMITH_TRACING"])
print("✅ LangSmith project:", os.environ["LANGSMITH_PROJECT"])

# 4. 🧠 LLM Calling

Before building an agent, test a normal LLM call.

### What is an LLM call?

```text
Python
  ↓
LLM API
  ↓
Model
  ↓
Response
```

At this stage there are **no tools**.


In [ ]:
# Create the LLM
from langchain_openai import ChatOpenAI

MODEL_NAME = "openai/gpt-4o-mini"

llm = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

print("✅ LLM initialized:", MODEL_NAME)

In [ ]:
# Simple LLM call
response = llm.invoke(
    "Explain AI agents in two simple sentences."
)

print(response.content)

# 5. 🔎 Tool 1 — Web Search

### When should Search be used?

Use Search for:

- latest information
- current information
- recent developments
- news
- time-sensitive questions

### Flow

```text
Agent
  ↓
web_search(query)
  ↓
SerpAPI
  ↓
Search results
  ↓
Agent
```

In [ ]:
# Web Search implementation
import requests
from langchain_core.tools import tool

def web_search_impl(query: str) -> str:
    api_key = os.environ.get("SERPAPI_API_KEY")

    if not api_key:
        return "ERROR: SERPAPI_API_KEY is missing."

    try:
        r = requests.get(
            "https://serpapi.com/search?engine=google",
            params={
                "engine": "google",
                "q": query,
                "api_key": api_key,
            },
            timeout=20,
        )

        if r.status_code != 200:
            return f"ERROR: Search API returned HTTP {r.status_code}."

        results = r.json().get("organic_results", [])

        if not results:
            return "ERROR: No search results were available."

        output = []

        for item in results[:5]:
            output.append(
                f"Title: {item.get('title', 'N/A')}\n"
                f"Snippet: {item.get('snippet', 'N/A')}\n"
                f"Link: {item.get('link', 'N/A')}"
            )

        return "\n\n".join(output)

    except Exception as e:
        return f"ERROR: Search failed — {e}"


@tool
def web_search(query: str) -> str:
    """Search the web for latest, current, recent, news, or time-sensitive information."""
    return web_search_impl(query)

print(web_search.name)
print(web_search.description)

In [ ]:
# Test Search directly
print(
    web_search.invoke({
        "query": "latest developments in generative AI"
    })
)

# 6. 🌤️ Tool 2 — Live Weather

### Purpose

Use Weather for:

- current temperature
- humidity
- wind speed
- weather condition
- city weather


In [ ]:
# Weather implementation
def weather_impl(city: str) -> str:
    api_key = os.environ.get("OPENWEATHER_API_KEY")

    if not api_key:
        return "ERROR: OPENWEATHER_API_KEY is missing."

    try:
        r = requests.get(
            "https://api.openweathermap.org/data/2.5/weather",
            params={
                "q": city,
                "appid": api_key,
                "units": "metric",
            },
            timeout=20,
        )

        if r.status_code == 404:
            return f"ERROR: City '{city}' was not found."

        if r.status_code != 200:
            return f"ERROR: Weather API returned HTTP {r.status_code}."

        data = r.json()

        return (
            f"City: {data.get('name', city)}\n"
            f"Temperature: {data['main']['temp']} °C\n"
            f"Feels Like: {data['main']['feels_like']} °C\n"
            f"Humidity: {data['main']['humidity']}%\n"
            f"Wind Speed: {data.get('wind', {}).get('speed', 'N/A')} m/s\n"
            f"Condition: {data['weather'][0]['description']}"
        )

    except Exception as e:
        return f"ERROR: Weather request failed — {e}"


@tool
def get_current_weather(city: str) -> str:
    """Get current temperature, humidity, wind speed, and weather condition for a city."""
    return weather_impl(city)

print(get_current_weather.name)
print(get_current_weather.description)

In [ ]:
# Test Weather directly
print(
    get_current_weather.invoke({
        "city": "Hyderabad"
    })
)

# 7. 📄 Tool 3 — PDF RAG

### RAG = Retrieval-Augmented Generation

```text
PDF
 ↓
Extract text
 ↓
Split text
 ↓
Create embeddings
 ↓
FAISS vector database
 ↓
Similarity search
 ↓
Relevant chunks
 ↓
LLM answer
```

In [ ]:
# Upload PDF
from google.colab import files
from pypdf import PdfReader

uploaded = files.upload()

pdf_name = next(
    (name for name in uploaded if name.lower().endswith(".pdf")),
    None,
)

if not pdf_name:
    raise ValueError("Please upload a PDF file.")

reader = PdfReader(pdf_name)

full_text = "\n".join(
    page.extract_text() or ""
    for page in reader.pages
)

if not full_text.strip():
    raise ValueError("No readable text was extracted from the PDF.")

print("✅ PDF:", pdf_name)
print("✅ Pages:", len(reader.pages))

In [ ]:
# Split PDF into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100,
)

documents = splitter.create_documents([full_text])

print("✅ Chunks:", len(documents))
print("\n--- Example chunk ---")
print(documents[0].page_content[:1000])

In [ ]:
# Create embeddings and FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(
    documents,
    embeddings,
)

print("✅ FAISS created")
print("Stored vectors:", vectorstore.index.ntotal)

In [ ]:
# Test similarity search
results = vectorstore.similarity_search(
    "What is the main problem statement?",
    k=3,
)

for i, doc in enumerate(results, 1):
    print(f"--- Retrieved chunk {i} ---")
    print(doc.page_content[:1200])
    print()

In [ ]:
# PDF RAG tool
@tool
def pdf_rag(question: str) -> str:
    """Search the uploaded PDF and return relevant passages for answering a document question."""

    try:
        docs = vectorstore.similarity_search(question, k=4)

        if not docs:
            return "ERROR: No relevant information was found in the PDF."

        return "\n\n".join(
            f"[PDF Chunk {i}]\n{doc.page_content}"
            for i, doc in enumerate(docs, 1)
        )

    except Exception as e:
        return f"ERROR: PDF retrieval failed — {e}"


print(pdf_rag.name)
print(pdf_rag.description)

# 8. 🧰 Tool Definitions

The agent needs to understand **what each tool does**.


| Tool | Agent should use it for |
|---|---|
| `web_search` | Latest/current/recent web information |
| `get_current_weather` | Current weather |
| `pdf_rag` | Questions about uploaded PDF |

In [ ]:
# All tools
tools = [
    web_search,
    get_current_weather,
    pdf_rag,
]

for t in tools:
    print("🔧", t.name)
    print("   ", t.description)
    print()

# 9. 🤖 Build One Agent

Now we combine:

```text
LLM
 +
Tools
 +
System Prompt
 =
One Multi-Tool Agent
```

### System prompt responsibility

The system prompt tells the agent:

- when to use Search
- when to use Weather
- when to use PDF RAG
- how to handle follow-up questions
- not to invent live or PDF information


In [ ]:
# Create the agent
from langchain.agents import create_agent

SYSTEM_PROMPT = """
You are a Multi-Tool AI Assistant.

TOOL SELECTION RULES:

1. WEB SEARCH:
Use web_search for latest, current, recent, news, or time-sensitive
information from the internet.

2. WEATHER:
Use get_current_weather for current temperature, humidity,
wind speed, weather conditions, or city weather.

3. PDF RAG:
Use pdf_rag for questions about the uploaded PDF.

4. FOLLOW-UP:
Use conversation history. If the user asks "What is the humidity there?",
identify the city from the previous weather conversation and use the
Weather tool for that same city.

5. DO NOT INVENT:
Do not invent live search/weather results or facts not found in the PDF.

6. FINAL ANSWER:
After receiving tool results, provide a clear concise answer.
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

print("✅ Agent created")
print("Tools:", [t.name for t in tools])

# 10. 🔄 Tool Calling — The Most Important Concept

When the user asks:

> **What is the current weather in Kolkata?**

the agent performs approximately:

```text
USER
 │
 ▼
LLM / AGENT
 │
 │ decides: weather tool
 ▼
get_current_weather(city="Kolkata")
 │
 ▼
Python executes function
 │
 ▼
OpenWeather API
 │
 ▼
Weather result
 │
 ▼
LLM / AGENT
 │
 ▼
Final human answer
```


# 11. 💬 Latest-10 Chat History

We implement:

```python
chat_history = result["messages"][-10:]
```

So the history never grows beyond 10 stored messages.

In [ ]:
import inspect
from langchain_core.messages import ToolMessage, AIMessage, HumanMessage

# Conversation runner
chat_history = []

def safe_slice_history(history_list, max_limit=10):
    """Slices chat history strictly to max_limit without splitting an AIMessage
    tool call from its corresponding ToolMessage response.
    """
    if len(history_list) <= max_limit:
        return history_list

    # Force clip to the maximum allowed limit first
    pruned_history = history_list[-max_limit:]

    # Inspect the leading message to avoid breaking LLM API sequence constraints
    while pruned_history:
        first_msg = pruned_history[0]

        # Check role type safely across object types or dictionaries
        role = getattr(first_msg, 'role', getattr(first_msg, 'type', None))
        if role is None and isinstance(first_msg, dict):
            role = first_msg.get('role')

        # Rule 1: A ToolMessage cannot start the conversation history window
        if role == 'tool' or isinstance(first_msg, ToolMessage):
            pruned_history.pop(0)
            continue

        # Rule 2: An AIMessage with tool calls cannot start the window if its response was cut
        if role == 'assistant' or isinstance(first_msg, AIMessage):
            has_tool_calls = bool(getattr(first_msg, 'tool_calls', None) or
                                  (isinstance(first_msg, dict) and first_msg.get('tool_calls')))
            if has_tool_calls:
                pruned_history.pop(0)
                continue

        # Valid entry point discovered (typically a HumanMessage)
        break

    return pruned_history

def ask_agent(question: str):
    global chat_history

    # Clean up the history before adding the new input sequence
    sliced_history = safe_slice_history(chat_history, 10)

    # Wrap the new user request structure
    messages = sliced_history + [HumanMessage(content=question)]

    # Invoke your compiled agent framework graph
    result = agent.invoke({
        "messages": messages
    })

    # Enforce strict memory limits safely on the returned pipeline messages
    chat_history = safe_slice_history(result["messages"], 10)

    # Safely isolate and process the text output string
    final_message = chat_history[-1]
    answer = (
        final_message.content
        if hasattr(final_message, "content")
        else (final_message.get("content", str(final_message)) if isinstance(final_message, dict) else str(final_message))
    )

    print("USER")
    print(question)

    print("\n ASSISTANT")
    print(answer)

    print("\n Stored message objects:", len(chat_history))

    return answer


In [ ]:
def clear_chat_history():
    global chat_history
    chat_history = []
    print("🧹 Chat history cleared.")

In [ ]:
# TEST — Clear history
print("Before clearing:", len(chat_history))

clear_chat_history()

assert len(chat_history) == 0

print("✅ PASS — Chat history successfully cleared.")

# 12.Test 1 — Current Web Information

### Question

> **What are the latest developments in generative AI?**

Expected tool:

```text
web_search
```


In [ ]:
# TEST 1 — Search
ask_agent(
    "What are the latest developments in generative AI?"
)

# 13.Test 2 — Current Weather

1.   List item
2.   List item



### Question

> **What is the current weather in Kolkata?**

Expected tool:

```text
get_current_weather
```

In [ ]:
# TEST 2 — Weather
ask_agent(
    "What is the current weather in Kolkata?"
)

# 14.Test 3 — PDF Question 1

### Question

> **According to the uploaded PDF, what is the main problem statement?**

Expected tool:

```text
pdf_rag
```

In [ ]:
# TEST 3 — PDF Question 1
ask_agent(
    "According to the uploaded PDF, short summary of the pdf?"
)

# 15. Test 4 — PDF Question 2

### Question

> **According to the uploaded PDF, what are the basic requirements?**

Expected tool:

```text
pdf_rag
```

In [ ]:
# TEST 4 — PDF Question 2
ask_agent(
    "According to the uploaded PDF, what are the basic requirements?"
)

# 16.Test 5 — Follow-Up

### Question

> **What is the humidity there?**

This tests conversation context.

The agent should understand that **“there”** refers to the city from the previous weather question.

Expected:

```text
Previous question → Kolkata
Follow-up → "there"
Agent → Kolkata
Agent → Weather tool
```

In [ ]:
# TEST 5 — Follow-up
ask_agent(
    "What is the humidity in there"
)

# 17. Test 6 — Prove Latest-10 Limit

We continue the conversation and verify that the stored history never exceeds 10 messages.

In [ ]:
# TEST 6 — Latest 10 messages
for i in range(1, 8):
    ask_agent(
        f"Give me one short observation about demonstration message {i}."
    )

    assert len(chat_history) <= 10

print("\n✅ PASS — History never exceeded 10 messages.")
print("Final stored message count:", len(chat_history))

# 18. 📊 LangSmith Tracing

### Why tracing?

LangSmith allows us to inspect the actual execution.

For example:

```text
User Question
     ↓
Agent
     ↓
Tool Selected
     ↓
Tool Input
     ↓
Tool Output
     ↓
Final Answer
```

In [ ]:
# LangSmith configuration check
print("LANGSMITH_TRACING:", os.environ.get("LANGSMITH_TRACING"))
print("LANGSMITH_PROJECT:", os.environ.get("LANGSMITH_PROJECT"))
print("API key configured:", bool(os.environ.get("LANGSMITH_API_KEY")))

print("\n✅ Run the agent tests above.")
print("Then open LangSmith and inspect the trace.")

# 19. ⚠️ Error Handling

The simple errors for:

- missing API key
- invalid city
- unavailable result

Our functions return readable `ERROR:` messages.

In [ ]:
# Error test — invalid city
print(
    weather_impl(
        "CityThatDefinitelyDoesNotExist999999"
    )
)

# 21. 🏗️ Final Architecture

```text
                         ┌──────────────────────┐
                         │     USER QUESTION    │
                         └──────────┬───────────┘
                                    │
                                    ▼
                         ┌──────────────────────┐
                         │    LANGCHAIN AGENT   │
                         │         + LLM        │
                         └──────────┬───────────┘
                                    │
                 ┌──────────────────┼──────────────────┐
                 ▼                  ▼                  ▼
          ┌────────────┐     ┌────────────┐     ┌────────────┐
          │ WEB SEARCH │     │  WEATHER   │     │  PDF RAG   │
          │  SerpAPI   │     │ OpenWeather│     │ FAISS      │
          └──────┬─────┘     └──────┬─────┘     └──────┬─────┘
                 │                  │                  │
                 └──────────────────┼──────────────────┘
                                    │
                                    ▼
                         ┌──────────────────────┐
                         │     TOOL RESULTS     │
                         └──────────┬───────────┘
                                    │
                                    ▼
                         ┌──────────────────────┐
                         │     FINAL ANSWER     │
                         └──────────────────────┘

              Supporting components:
              ├── Latest 10 messages
              ├── Secure API keys
              ├── Error handling
              └── LangSmith tracing
```

In [ ]:
# AGENT HEALTH CHECK
checks = {
    "LLM initialized": "llm" in globals(),
    "Search tool": "web_search" in globals(),
    "Weather tool": "get_current_weather" in globals(),
    "PDF RAG tool": "pdf_rag" in globals(),
    "Agent": "agent" in globals(),
    "FAISS vector store": "vectorstore" in globals(),
    "History <= 10": len(chat_history) <= 10,
    "LangSmith enabled": os.environ.get("LANGSMITH_TRACING") == "true",
}

print("========== AGENT HEALTH CHECK ==========")

for name, passed in checks.items():
    print(("✅" if passed else "❌"), name)

print("==========================================")